# Mini practice 1 · Is this sample fit for this platform?

**Genomic Data Processing · Day 2 (10/13) · about 15 minutes · practice slide 25**

Twelve tubes arrive at a sequencing core with a sample sheet. Before anything goes on a
machine, someone decides, tube by tube: **PASS**, **REVIEW**, **FAIL** or **BLOCKED**, and
writes down why. In Part A that someone is you. In Part B you look at what three platforms
return from the same blood DNA.

One idea runs through both parts:

> A genomics sample is **specimen + molecule + metadata**, and the sample sets the ceiling for
> what any platform can read. **You can clean DNA; you cannot make it longer.**

### What is real and what is synthetic
| Part of the package | Status |
|---|---|
| `hbb_window.hg38.fa`: hg38 chr11:5,218,771–5,278,770 | **Real** UCSC sequence, the same 60-kb β-globin window as Day 1 |
| `genes.hg38.bed`: β-globin genes and exons | **Real** NCBI RefSeq Select models, from Day 1 |
| Input, length and purity numbers in `platform_requirements.tsv` | From the vendor document named in its `source` column, otherwise marked **teaching threshold** |
| The 12 samples, every read, the PAF alignments | **Synthetic**: simulated, no real person |

Nothing in this notebook is a clinical result.

### How to work
* Run the cells from top to bottom (Shift+Enter). Each one finishes in about a second.
* **Predict, then run.** When a cell asks for a prediction, say it or write it down first.
* Each step says **what changes** and **how to check it**. If a cell will not run, the verified
  answer is in `checkpoints/`: load it and keep going.

In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA  = Path("data/practice1")     # open this notebook from the day2-3-samples/ folder
READS = DATA / "reads"
CHECK = Path("checkpoints")
OUT   = Path("outputs")
OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "legend.frameon": False, "axes.spines.top": False,
                     "axes.spines.right": False})

print("data :", sorted(p.name for p in DATA.iterdir() if p.is_file()))
print("reads:", sorted(p.name for p in READS.iterdir()))

---
## Part A · From sample sheet to platform decision

*Concept slides: "Measure the molecule before you sequence" (slide 6), "Metadata: the sample's
passport" (slide 7), "Match the sample to the platform" (slide 22).*

### A1 · Read the sample sheet
Each row is one tube, and it carries three kinds of information:

* **specimen**: where the material came from (blood, saliva, an FFPE tumour block, plasma, cells)
* **molecule**: what was extracted (DNA, RNA, or intact cells that have not been extracted yet)
* **metadata**: what the study wants (`requested_assay`, `study_question`) and whether we may
  use the tube at all (`consent_genomic_research`)

`NA` means *not measured*. It never means zero.

**Predict, then run.** One tube must not be processed whatever its quality numbers say.
Which column will tell you?

In [ ]:
sheet = pd.read_csv(DATA / "sample_sheet.tsv", sep="\t")
print(sheet.shape[0], "tubes,", sheet.shape[1], "columns")
sheet[["sample_id", "subject_id", "specimen", "molecule", "requested_assay",
       "study_question", "matched_normal_id", "consent_genomic_research"]]

It is `consent_genomic_research`: **S12** says `no`. Keep it in mind; we come back to it.

Now the measurements. What each column tells you:

| column | what it measures | can it be fixed? |
|---|---|---|
| `conc_ng_per_ul` × `volume_ul` | how much material is in the tube (ng) | sometimes: re-extract, concentrate, pool |
| `a260_280`, `a260_230` | absorbance ratios. Protein or phenol carry-over lowers 260/280; guanidine salts, phenol or carbohydrates lower 260/230 | **yes**, with a clean-up |
| `din`, `rin` | DNA / RNA integrity number, from 1 (degraded) to 10 (intact) | no |
| `dv200` | % of RNA fragments longer than 200 nucleotides, used when RNA is too degraded for RIN to help | no |
| `modal_fragment_kb` | the most common DNA fragment length | **no**: a read cannot be longer than the molecule |
| `cell_count` | intact cells, for protocols that extract the DNA themselves | recollect |

In [ ]:
sheet[["sample_id", "specimen", "molecule", "conc_ng_per_ul", "volume_ul", "a260_280",
       "a260_230", "din", "rin", "dv200", "modal_fragment_kb", "cell_count"]]

### A2 · What each platform needs
Read the `source` column before you trust a number. A threshold is only as good as the
document it came from; anything without a vendor source is labelled *teaching threshold*.

In [ ]:
req = pd.read_csv(DATA / "platform_requirements.tsv", sep="\t").set_index("assay")
req[["platform", "kit_or_chemistry", "molecule", "min_input", "input_unit",
     "min_fragment_kb", "integrity_rule", "purity_rule"]]

In [ ]:
for assay, r in req.iterrows():
    print(f"{assay}\n  notes : {r['notes']}\n  source: {r['source']}")

### A3 · Total mass = concentration × volume
Kits ask for nanograms; a tube label gives ng/µL. Worked example for **S08**:
30 ng/µL × 20 µL = **600 ng**. PacBio asks for 500 ng, so S08 has enough, with little to spare.

**What changes:** the sheet gains one column, `total_ng`.
**How to check:** S08 must show 600, and the cell line S07 must show `NaN`, because cells are
counted, not weighed.

In [ ]:
sheet["total_ng"] = sheet["conc_ng_per_ul"] * sheet["volume_ul"]

s08 = sheet[sheet.sample_id == "S08"].iloc[0]
assert s08.total_ng == 600
sheet[["sample_id", "specimen", "conc_ng_per_ul", "volume_ul", "total_ng", "requested_assay"]]

### A4 · The decision rules, in the order a core facility applies them
1. **Consent.** No consent: **BLOCKED**. Stop here; do not even look at the quality numbers.
2. **Right molecule, enough cells.** The assay must receive the molecule it expects. For
   cell-input protocols, compare `cell_count` with the kit minimum.
3. **Enough mass.** Below the minimum: **REVIEW** if you have at least half, **FAIL** if less.
4. **Long or intact enough.** Below the integrity rule: **FAIL**. *This cannot be repaired.*
5. **Clean enough.** Purity outside the range: **REVIEW** (clean up, re-measure). *Never FAIL.*
6. **Somatic question on a tumour.** No matched normal from the same person: add a **REVIEW**.

Otherwise the tube is a **PASS**. One tube can collect several reasons; its decision is the
worst of them.

**Predict, then run.** Write down your call for **S03**, **S06**, **S11** and **S12** before
you run the cell after next.

In [ ]:
# purity windows (low, high) copied from the purity_rule column; None = no upper limit
PURITY = {"DNA": {"a260_280": (1.8, 2.0), "a260_230": (2.0, 2.2)},
          "RNA": {"a260_280": (1.9, 2.1), "a260_230": (1.8, None)}}
SUBJECT = dict(zip(sheet.sample_id, sheet.subject_id))      # which person each tube is from

def decide(s, r):
    """Decide one tube. s = one row of the sample sheet, r = one row of the requirements.
    Returns (decision, list of reasons)."""
    # 1. Consent first: without consent we never look at quality
    if s["consent_genomic_research"] != "yes":
        return "BLOCKED", ["BLOCKED: no consent for genomic research (do not process, do not assess quality)"]

    # 2. The assay must receive the molecule it expects (DNA, RNA or intact cells)
    if s["molecule"] != r["molecule"]:
        return "FAIL", [f"FAIL: assay needs {r['molecule']} but the tube holds {s['molecule']}"]
    reasons = []

    # 2-3. Enough material? Cells are counted; DNA and RNA are weighed (ng = conc x volume)
    unit = r["input_unit"]
    have = s["cell_count"] if unit == "cells" else s["conc_ng_per_ul"] * s["volume_ul"]
    need = r["min_input"]
    if have < need:
        level = "REVIEW" if have >= 0.5 * need else "FAIL"
        reasons.append(f"{level}: input {have:,.0f} {unit} is {have / need:.0%} of the {need:,.0f} {unit} minimum")

    # 4. Long or intact enough? A rule reads like "modal_fragment_kb >= 15". Cannot be repaired.
    if pd.notna(r["integrity_rule"]):
        metric, threshold = r["integrity_rule"].split(" >= ")
        threshold = float(threshold)
        value = s[metric]
        if pd.isna(value):
            reasons.append(f"REVIEW: {metric} not measured")
        elif value < threshold:
            reasons.append(f"FAIL: {metric} {value:g} < {threshold:g} required (cannot be repaired)")

    # 5. Clean enough? A clean-up fixes purity, so this is REVIEW, never FAIL
    for ratio, (low, high) in PURITY.get(s["molecule"], {}).items():
        value = s[ratio]
        if pd.isna(value):
            reasons.append(f"REVIEW: {ratio} not measured")
        elif value < low:
            reasons.append(f"REVIEW: {ratio} {value:.2f} below {low:.2f} (clean up, re-measure)")
        elif high is not None and value > high:
            reasons.append(f"REVIEW: {ratio} {value:.2f} above {high:.2f} (clean up, re-measure)")

    # 6. A somatic question on a tumour needs a matched normal from the same person
    if "tumour" in s["specimen"] and "somatic" in s["study_question"].lower():
        normal_id = s["matched_normal_id"]
        if pd.isna(normal_id):
            reasons.append("REVIEW: tumour sample for a somatic question but no matched normal")
        elif SUBJECT.get(normal_id) != s["subject_id"]:
            reasons.append(f"REVIEW: matched normal {normal_id} is not from the same subject")

    levels = [reason.split(":")[0] for reason in reasons]
    decision = "FAIL" if "FAIL" in levels else "REVIEW" if "REVIEW" in levels else "PASS"
    return decision, reasons

print("decide() is ready")

Look at steps 4 and 5 side by side. Both compare a number with a threshold, but only step 4
can return FAIL. That one difference *is* the lecture's message: a clean-up fixes purity;
nothing fixes length.

Now apply `decide()` to every tube, each against the assay it requested.
**What changes:** a new table with one row per tube. **How to check:** 12 rows, and every
non-PASS row carries at least one reason.

In [ ]:
rows = []
for _, s in sheet.iterrows():
    decision, reasons = decide(s, req.loc[s["requested_assay"]])
    rows.append({"sample_id": s["sample_id"], "requested_assay": s["requested_assay"],
                 "total_ng": s["total_ng"], "decision": decision,
                 "reasons": "; ".join(reasons) if reasons else "-"})
decisions = pd.DataFrame(rows)

assert len(decisions) == 12
assert (decisions.loc[decisions.decision != "PASS", "reasons"] != "-").all()
for r in decisions[decisions.decision != "PASS"].itertuples():
    print(f"{r.sample_id} {r.decision}")
    for reason in r.reasons.split("; "):
        print("    " + reason)
decisions[["sample_id", "requested_assay", "total_ng", "decision"]]

In [ ]:
tally = decisions["decision"].value_counts()
print(tally.to_string())
assert tally.to_dict() == {"PASS": 6, "FAIL": 3, "REVIEW": 2, "BLOCKED": 1}, \
    "different tally: compare your table with checkpoints/practice1_decisions.tsv"

*If `decide()` will not run, load the verified table and continue:*
`decisions = pd.read_csv(CHECK / "practice1_decisions.tsv", sep="\t")`

### Checkpoint A · read three rows aloud
* **S03** (FFPE tumour DNA → PacBio HiFi) fails twice: 240 ng is under half the input, and the
  fragments are 0.35 kb when the library needs 15–20 kb. More FFPE sections could fix the mass;
  nothing fixes the length. It also lacks a matched normal for a somatic question.
* **S06** (plasma cfDNA → ONT long reads). Cell-free DNA is about 167 bp because it was cut
  between nucleosomes before it left the cell. No platform can read across 50 kb from it, and
  20 ng is 2% of the 1 µg the kit asks for.
* **S12** has excellent numbers and is **BLOCKED**. Consent is checked before quality, and a
  blocked tube's quality data are not used for anything.

> **Q.** S02 and S11 are both REVIEW. Is it the same problem?
> **A.** No. S02 is *dirty* (A260/230 = 1.45, typical of salt or guanidine carry-over): a bead
> clean-up and a re-measurement fix it. S11 is *short of mass* (300 of 500 ng): re-quantify with
> a fluorometer, concentrate, recollect, or move to an assay that needs less.

### A5 · From a decision to a suggestion
Rules tell you when to stop. Choosing what to do instead needs the study question, so the
suggestions below are written by a person and kept *next to* the rule-based decision, never
mixed into it. S06 shows why: mutation detection and 50-kb phasing lead to different specimens.

**What changes:** a `suggested_alternative` column, then the table is saved.
**How to check:** the printed line says `True` when your file equals the checkpoint.

In [ ]:
ALTERNATIVE = {
    "S02": "Bead clean-up, re-measure A260/230, then run ILMN_PCRFREE_WGS as requested",
    "S03": "ILMN_WES or a panel with an FFPE-compatible prep, plus a matched normal (e.g. blood) from P003",
    "S06": "Mutation detection: Illumina cfDNA targeted panel. Phasing: collect blood and extract long DNA",
    "S10": "RNA_CAPTURE_FFPE: DV200 62% (medium quality) needs 20-40 ng; see the re-check",
    "S11": "Re-quantify (fluorometer) and concentrate; or recollect as blood; or switch to ILMN_PCRFREE_WGS (resolves fewer SVs)",
    "S12": "Do not process; refer back to the study team (consent)",
}
decisions["suggested_alternative"] = [ALTERNATIVE.get(sid, "-") for sid in decisions.sample_id]
decisions.to_csv(OUT / "practice1_decisions.tsv", sep="\t", index=False, na_rep="NA")

mine = pd.read_csv(OUT / "practice1_decisions.tsv", sep="\t")
print("matches checkpoint:", mine.equals(pd.read_csv(CHECK / "practice1_decisions.tsv", sep="\t")))
decisions[["sample_id", "decision", "suggested_alternative"]]

### A6 · Same tube, better-matched assay
**S10** is FFPE tumour RNA with RIN 2.3. Poly(A) selection grabs each transcript by its 3′ tail,
so on fragmented RNA you keep only the pieces nearest the tail (3′ bias). Capture-based RNA-seq
uses probes along the whole transcript and tolerates fragments. For degraded RNA the useful
number is **DV200**, not RIN.

**Predict, then run.** S10 against `RNA_CAPTURE_FFPE`: PASS or not? And S03 against the exome
(`ILMN_WES`), which accepts short FFPE DNA?

In [ ]:
def dv200_band(dv200):
    """Recommended input for capture RNA-seq by DV200 (Illumina technical note)."""
    if dv200 > 70:
        return "high quality: 20 ng"
    if dv200 >= 50:
        return "medium quality: 20-40 ng"
    if dv200 >= 30:
        return "low quality: 40-100 ng"
    return "too degraded: not recommended"

def tube(sample_id):
    return sheet[sheet.sample_id == sample_id].iloc[0]

for sample_id, assay in [("S10", "RNA_POLYA"), ("S10", "RNA_CAPTURE_FFPE"),
                         ("S03", "PACBIO_HIFI_WGS"), ("S03", "ILMN_WES")]:
    decision, reasons = decide(tube(sample_id), req.loc[assay])
    print(f"{sample_id} on {assay:<16} -> {decision:<6} {'; '.join(reasons)}")

s10 = tube("S10")
print(f"\nS10: DV200 {s10.dv200:g}% -> {dv200_band(s10.dv200)}; the tube holds {s10.total_ng:g} ng")
assert decide(s10, req.loc["RNA_CAPTURE_FFPE"])[0] == "PASS"

### Checkpoint A6
* **S10:** FAIL → **PASS** by changing the assay, not the tube. DV200 62% is *medium quality*,
  for which 20–40 ng is recommended, and the tube holds 225 ng.
* **S03:** FAIL → **REVIEW** on the exome. The mass and length problems disappear: the exome
  needs only 100 ng, and short-read libraries are built from short fragments anyway. Two flags
  remain: a clean-up for A260/230 = 1.95, and the missing matched normal. A platform cannot fix
  the study design.

---
## Part B · What each platform returns

*Concept slide: "Side by side: which platform for which question?" (slide 21).*

We simulated **S08**, the blood DNA from Part A (modal fragment 22 kb), sequenced three ways
over the real 60-kb β-globin window: **Illumina 2×150**, **PacBio HiFi** and **ONT ligation**.
Same molecule, same region, roughly the same number of bases. The question: which reads cross
an 8-kb stretch from inside *HBB* to inside *HBD*, the kind of span you need to describe a
structural variant there? This is a teaching target; we make no clinical claim about it.

### B1 · The reference window and the target
The FASTA header `>chr11:5218771-5278770` is a 1-based, closed address (the way a genome
browser writes it). In BED terms the window is `[5218770, 5278770)`, so character `i` of the
sequence string (counting from 0) is chr11 BED position `5218770 + i`.

In [ ]:
def read_fasta(path):
    name, chunks = None, []
    for line in open(path):
        line = line.rstrip()
        if line.startswith(">"):
            name = line[1:]
        else:
            chunks.append(line)
    return name, "".join(chunks)

window_name, window = read_fasta(DATA / "hbb_window.hg38.fa")
WIN_START0 = 5218770                        # BED start of the window on chr11
print(window_name, f"{len(window):,} bp")
assert len(window) == 60_000

# the 18 bases you met on Day 1: hg38 chr11:5,226,995-5,227,012 (1-based, closed)
day1 = window[5226995 - 1 - WIN_START0 : 5227012 - WIN_START0]
print("chr11:5,226,995-5,227,012 =", day1)
assert day1 == "CTTCTCCTCAGGAGTCAG"

In [ ]:
target = pd.read_csv(DATA / "target_window.bed", sep="\t", header=None,
                     names=["chrom", "start", "end", "name"])
T_START, T_END = int(target["start"][0]), int(target["end"][0])
print(f"target {target['name'][0]}: chr11 [{T_START:,}, {T_END:,}) = {T_END - T_START:,} bp")

genes = pd.read_csv(DATA / "genes.hg38.bed", sep="\t")
inside = genes[(genes.chromStart < T_END) & (genes.chromEnd > T_START)]
inside[["name", "chromStart", "chromEnd", "strand"]]

The target holds the whole of *HBB* exon 1 and *HBD* exons 2–3, the edge of *HBB* exon 2,
and the *HBB* promoter.

### B2 · One FASTQ record is four lines
*Theory slide: "Phred quality: how sure is each base?" (slide 14).*

```
@read_name        line 1: '@' then the read name
ACGTTGCA...       line 2: the bases
+                 line 3: a separator
??+5@CC>...       line 4: one quality character per base
```
Each quality character encodes a **Phred score**: Q = (ASCII code − 33), and the chance the base
is wrong is p = 10^(−Q/10). So Q10 = 1 error in 10, Q20 = 1 in 100, Q30 = 1 in 1,000,
Q40 = 1 in 10,000. Real read names are longer (PacBio: `movie/hole/ccs`; ONT: a random ID);
ours are short so that you can read them.

In [ ]:
FASTQ = {"illumina_2x150": [READS / "illumina_R1.fastq.gz", READS / "illumina_R2.fastq.gz"],
         "pacbio_hifi":    [READS / "pacbio_hifi.fastq.gz"],
         "ont_ligation":   [READS / "ont_ligation.fastq.gz"]}

def read_fastq(path):
    """Yield (name, bases, quality) for each 4-line FASTQ record."""
    with gzip.open(path, "rt") as fh:
        while True:
            header = fh.readline().rstrip()
            if not header:
                return
            bases = fh.readline().rstrip()
            fh.readline()                           # the '+' separator line
            quality = fh.readline().rstrip()
            assert header.startswith("@") and len(bases) == len(quality)
            yield header[1:], bases, quality

name, bases, quality = next(read_fastq(FASTQ["illumina_2x150"][0]))
print("name   :", name, f"({len(bases)} bases)")
print("bases  :", bases[:40], "...")
print("quality:", quality[:40], "...")
print("Q      :", [ord(c) - 33 for c in quality[:12]], "...")

### B3 · Read N50: your turn
*Theory slide: "Read length: mean versus N50" (slide 20).*

The **read N50** is the length L such that reads of length L or longer hold at least half of
all the bases. It describes the read a typical *base* sits in, which is what matters for
spanning, rather than the typical read.

Worked example: lengths 2, 3, 4, 5, 6 (total 20, half = 10). Longest first: 6 (running total 6),
then 5 (running total 11, now ≥ 10). **N50 = 5**, while the mean is 4.

Write `read_n50` from the three comments; the two `assert` lines check you. A working version is
filled in so the notebook runs end to end: cover it and try first.

In [ ]:
def read_n50(lengths):
    """Length L such that reads of length >= L hold at least half of all bases."""
    # 1. sort the lengths from longest to shortest
    # 2. walk down the list, keeping a running total of bases
    # 3. return the first length at which the running total reaches half of all bases
    ordered = sorted(lengths, reverse=True)
    half = sum(ordered) / 2
    running = 0
    for length in ordered:
        running += length
        if running >= half:
            return length

assert read_n50([2, 3, 4, 5, 6]) == 5
assert read_n50([150] * 10) == 150         # equal lengths: the N50 is that length
print("read_n50 passes both checks")

Now read every record of every file and keep, per read, its length and two sums over its bases:
the Q values and the error probabilities. **What changes:** a table with one row per read.
**How to check:** 4,000 Illumina reads (2,000 pairs), 40 HiFi reads, 50 ONT reads.

In [ ]:
records = []
for platform, paths in FASTQ.items():
    for path in paths:
        for name, bases, quality in read_fastq(path):
            q = np.frombuffer(quality.encode(), dtype=np.uint8).astype(int) - 33
            records.append({"platform": platform, "read": name, "length": len(bases),
                            "sum_q": q.sum(), "sum_error": (10 ** (-q / 10)).sum()})
reads = pd.DataFrame(records)

print(reads.groupby("platform", sort=False).size().to_dict())
reads.groupby("platform", sort=False)["length"].describe()[["count", "mean", "min", "max"]]

### B4 · Average the error probabilities, not the Q values
*Theory slide: "Phred quality: how sure is each base?" (slide 14).*

Q is a logarithm, so averaging Q values hides bad bases. Four bases at Q40, Q40, Q40 and Q10:

* mean Q = 32.5, which sounds like about 1 error in 1,800 bases;
* mean error probability = (0.0001 + 0.0001 + 0.0001 + 0.1) / 4 = 0.025, which is 1 in 40.

The one bad base dominates. The honest summary is the mean of p (convert it back to a Q if you
like). **Predict, then run:** which platform does the naive average flatter most?

In [ ]:
q = np.array([40, 40, 40, 10])
p = 10 ** (-q / 10)
print(f"worked example: mean Q = {q.mean():.1f} -> error {10 ** (-q.mean() / 10):.5f}; "
      f"mean error = {p.mean():.5f} -> Q{-10 * np.log10(p.mean()):.1f}")

totals = reads.groupby("platform", sort=False)[["length", "sum_q", "sum_error"]].sum()
totals["mean_Q_naive"] = totals.sum_q / totals.length
totals["error_if_you_average_Q"] = 10 ** (-totals.mean_Q_naive / 10)
totals["mean_error"] = totals.sum_error / totals.length
totals["honest_Q"] = -10 * np.log10(totals.mean_error)
totals[["mean_Q_naive", "error_if_you_average_Q", "mean_error", "honest_Q"]].round(5)

**PacBio HiFi** is flattered most: Q43 by the naive average, about Q31 honestly. Most of its bases
carry very high Q values, and its few likely errors sit in homopolymers (runs such as `AAAAAA`),
at bases with much lower Q. Averaging Q values lets the many excellent bases hide the few
doubtful ones; averaging error probabilities does not.

### B5 · Where did each read land? The PAF file
PAF is a plain tab-separated table with one line per alignment. Its first 12 columns:

| # | column | meaning |
|---|---|---|
| 1 | `qname` | read name |
| 2 | `qlen` | read length |
| 3–4 | `qstart`, `qend` | aligned part of the read (0-based, end excluded) |
| 5 | `strand` | `+` the read matches the reference as written; `-` it matches the reverse complement |
| 6 | `tname` | reference sequence name (`chr11`) |
| 7 | `tlen` | reference length (hg38 chr11 = 135,086,622 bp) |
| 8–9 | `tstart`, `tend` | aligned part of the reference: **0-based, end excluded, exactly like BED** |
| 10 | `nmatch` | bases that match the reference |
| 11 | `alnlen` | alignment length: matches, mismatches, insertions and deletions |
| 12 | `mapq` | mapping quality; **255 means not available** |

Our PAF was written from the simulation's truth rather than by an aligner, which is why every
`mapq` is 255. With real reads you would run, for example,
`minimap2 -x map-hifi ref.fa reads.fq.gz > aln.paf` (`map-ont` for nanopore, `sr` for short
reads). The coordinates are real chr11 positions, not positions inside our 60-kb string.

In [ ]:
PAF_COLUMNS = ["qname", "qlen", "qstart", "qend", "strand", "tname", "tlen",
               "tstart", "tend", "nmatch", "alnlen", "mapq"]
paf = pd.read_csv(READS / "alignments.paf", sep="\t", header=None, names=PAF_COLUMNS)
prefix_to_platform = {"ilmn": "illumina_2x150", "hifi": "pacbio_hifi", "ont": "ont_ligation"}
paf["platform"] = paf.qname.str.split("_").str[1].map(prefix_to_platform)

assert paf.platform.notna().all() and len(paf) == len(reads)
assert ((paf.tstart >= WIN_START0) & (paf.tend <= WIN_START0 + len(window))).all()
print(len(paf), "alignments, all inside the window")
paf.groupby("platform", sort=False).head(2)

A read **spans** the target only if it starts at or before the target's start *and* ends at or
after the target's end: `tstart <= 5,226,000` and `tend >= 5,234,000`.

**Predict, then run.** How many Illumina reads span 8 kb? How many HiFi reads?

In [ ]:
paf["spans_target"] = (paf.tstart <= T_START) & (paf.tend >= T_END)
spanning = paf.groupby("platform", sort=False)["spans_target"].sum()
spanning

### B6 · One summary row per platform
*Theory slides: "Depth: how much sequencing do I need?" (slide 23) and "Coverage is uneven: the Poisson
view" (slide 24).*

Depth here is approximate: total bases ÷ window length (60,000 bp).
**What changes:** `outputs/practice1_read_stats.tsv` is written.
**How to check:** it equals the checkpoint; Illumina spans 0 reads, HiFi and ONT at least 3 each.
*If an earlier Part B cell will not run, the verified table is
`pd.read_csv(CHECK / "practice1_read_stats.tsv", sep="\t")`.*

In [ ]:
by_platform = reads.groupby("platform", sort=False)["length"]
stats = pd.DataFrame({
    "reads": by_platform.size(),
    "total_bases": by_platform.sum(),
    "mean_length": by_platform.mean().round(1),
    "max_length": by_platform.max(),
    "read_n50": by_platform.apply(read_n50),
    "mean_error_rate": (totals.sum_error / totals.length).round(6),
    "mean_accuracy_pct": (100 * (1 - totals.sum_error / totals.length)).round(3),
    "approx_depth_x": (by_platform.sum() / len(window)).round(2),
    "reads_spanning_target": spanning,
}).rename_axis("platform").reset_index()
stats.to_csv(OUT / "practice1_read_stats.tsv", sep="\t", index=False)

mine = pd.read_csv(OUT / "practice1_read_stats.tsv", sep="\t")
print("matches checkpoint:", mine.equals(pd.read_csv(CHECK / "practice1_read_stats.tsv", sep="\t")))
span = stats.set_index("platform")["reads_spanning_target"]
assert span["illumina_2x150"] == 0 and span["pacbio_hifi"] >= 3 and span["ont_ligation"] >= 3
stats

### Checkpoint B · same depth, different answers
All three platforms read this window to roughly 10–12×, yet only long reads cross the 8-kb
target. Depth tells you how much you read; length tells you what you can connect. HiFi reads are
long *and* accurate (about 99.9% per base). The ONT reads here are about 99% accurate per base,
with errors concentrated in homopolymers, and their lengths vary far more.

Two honest limits of this simulation:
* Our ONT reads stop below 58 kb only because the window is 60 kb. Real ultra-long libraries
  (SQK-ULK114, started from cells like S07) reach read N50s of 50–100+ kb and single reads of
  several megabases.
* S08's DNA has a modal fragment of 22 kb, and the ONT read N50 sits just below it. On a
  ligation kit, read length is fragment length: the tube, not the machine, set the ceiling.

### B7 · Picture it: read lengths on a log scale
A log axis lets 150-bp and 50-kb reads share one plot. The dashed line marks the 8-kb target: a
read to its left cannot span the target, whatever the depth.

In [ ]:
order = ["illumina_2x150", "pacbio_hifi", "ont_ligation"]
bins = np.logspace(np.log10(100), np.log10(60_000), 41)
fig, axes = plt.subplots(3, 1, figsize=(7.5, 6), sharex=True)
for ax, platform in zip(axes, order):
    lengths = reads.loc[reads.platform == platform, "length"]
    ax.hist(lengths, bins=bins, color="#4C72B0")
    ax.axvline(T_END - T_START, color="0.2", linestyle="--", linewidth=1)
    ax.set_ylabel("reads")
    ax.set_title(f"{platform}: {len(lengths):,} reads, read N50 {read_n50(lengths):,} bp, "
                 f"{int(spanning[platform])} span the target", loc="left")
axes[0].annotate("8-kb target", xy=(T_END - T_START, 0.75), xycoords=("data", "axes fraction"),
                 xytext=(6, 0), textcoords="offset points", fontsize=8)
axes[-1].set_xscale("log")
axes[-1].set_xticks([100, 1_000, 10_000, 50_000])
axes[-1].set_xticklabels(["100 bp", "1 kb", "10 kb", "50 kb"])
axes[-1].set_xlabel("read length (log scale)")
fig.suptitle("Only the long reads reach the 8-kb target (S08 blood DNA, same 60-kb window)", fontsize=10)
fig.tight_layout()
fig.savefig(OUT / "read_length_hist.png", dpi=150)
plt.show()

### B8 · Optional, if you have two minutes: draw the reads over the target
Each line is one read at its PAF coordinates; the shaded band is the target. Dark lines span it.
For Illumina only the reads near the target are drawn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True)
for ax, platform in zip(axes, order):
    hits = paf[paf.platform == platform].sort_values("tstart")
    if platform == "illumina_2x150":
        hits = hits[(hits.tend > T_START - 2_000) & (hits.tstart < T_END + 2_000)]
    for y, r in enumerate(hits.itertuples()):
        ax.plot([r.tstart / 1e6, r.tend / 1e6], [y, y], color="0.1" if r.spans_target else "0.65",
                linewidth=1.6 if r.spans_target else 0.7)
    ax.axvspan(T_START / 1e6, T_END / 1e6, color="#4C72B0", alpha=0.15)
    ax.set_title(f"{platform}: {int(hits.spans_target.sum())} of {len(hits):,} drawn span it", loc="left")
    ax.set_xlabel("chr11 position (Mb, hg38)")
    ax.set_yticks([])
axes[0].set_ylabel("reads, sorted by start")
fig.tight_layout()
fig.savefig(OUT / "target_spanning_reads.png", dpi=150)
plt.show()

---
### What you produced
| file | what it shows |
|---|---|
| `outputs/practice1_decisions.tsv` | one decision per tube, every reason kept, suggestions kept separate |
| `outputs/practice1_read_stats.tsv` | reads, bases, N50, honest error rate, depth and spanning reads per platform |
| `outputs/read_length_hist.png` | why depth alone cannot answer a spanning question |
| `outputs/target_spanning_reads.png` | (optional) the spanning reads themselves |

**Say it in one sentence.** *"I would not put S03 on PacBio, because its fragments (0.35 kb) are
far shorter than the 15–20 kb library; I checked the fragment size, which no clean-up can
change; I would run an FFPE exome with a matched normal instead."* Now say one for a tube of
your choice.